# Mini-GPT: Trainiere deinen eigenen Transformer

ChatGPT, Gemini, Claude … sie alle basieren auf derselben Architektur: dem **Transformer**.
In diesem Notebook baust und trainierst du selbst einen kleinen **GPT** (*Generative Pre-trained Transformer*) – auf einem Text, den **du** auswählst.

Dein Modell lernt dabei nur eine einzige Aufgabe:

> **Welches Zeichen kommt als nächstes?**

Klingt simpel – aber wenn man das oft genug wiederholt, entsteht Text, der so klingt wie dein Datensatz.

### Ablauf

| Schritt | Was passiert |
|---|---|
| 0 | Setup – GPU oder CPU? |
| 1 | Datensatz auswählen |
| 2 | Text in Zahlen umwandeln (Tokenisierung) |
| 3 | Trainingsbeispiele bauen |
| 4 | Das Modell bauen |
| 5 | Training |
| 6 | Text generieren |
| 7 | Ins Modell hineinschauen |
| 8 | Modell speichern |
| 9 | Challenges & Reflexion |

### So arbeitest du mit diesem Notebook
- Führe die Zellen **der Reihe nach** aus: Zelle anklicken → **Shift + Enter**.
- **Aufgabe**-Abschnitte sind für dich: Probiere etwas aus und notiere deine Beobachtungen in den **Deine Antwort**-Zellen darunter.
- Die Zelle **Hilfsfunktionen** enthält Hilfscode (z. B. für Diagramme). Den musst du nicht verstehen.
- Wenn etwas hängt oder kaputt ist: **Kernel → Restart Kernel** und die Zellen noch einmal von oben ausführen.

## 0. Setup

Neuronale Netze trainieren ist vor allem **sehr viel Rechnen mit Matrizen**. Eine Grafikkarte (GPU) kann tausende Rechnungen gleichzeitig ausführen und ist dafür viel schneller als der Prozessor (CPU).
Die nächste Zelle prüft, ob eine NVIDIA-GPU verfügbar ist – sonst wird die CPU verwendet.

In [ ]:
%matplotlib inline
import math
import time
import shutil

import torch
import torch.nn as nn
import torch.nn.functional as F
import ipywidgets as widgets
from IPython.display import display

torch.manual_seed(42)

if torch.cuda.is_available():
    device = "cuda"
    print(f"NVIDIA-GPU gefunden: {torch.cuda.get_device_name(0)} – das Training läuft auf der GPU.")
else:
    device = "cpu"
    print("Keine nutzbare NVIDIA-GPU gefunden – das Training läuft auf der CPU.")
    print("   Das ist langsamer, funktioniert aber. Wähle in Schritt 4 eine kleine Modellgröße.")
    if shutil.which("nvidia-smi"):
        print("\nAuf diesem Computer ist eine NVIDIA-GPU, aber PyTorch wurde ohne GPU-Unterstützung installiert.")
        print("   Installiere die GPU-Version im Terminal und starte danach den Kernel neu:")
        print("   pip install torch --index-url https://download.pytorch.org/whl/cu128")

In [ ]:
# Hilfsfunktionen für Datensätze und Visualisierungen.
#    Du musst diesen Code nicht verstehen – einfach ausführen!
import html
import re
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, clear_output

_repo_data = Path.cwd().parent / "data"
DATA_DIR = _repo_data if _repo_data.is_dir() else Path.cwd() / "data"
DATA_DIR.mkdir(exist_ok=True)

MIN_CHARS = 10_000
MAX_CHARS = 5_000_000

PRESETS = {
    "Shakespeare – Theaterstücke (Englisch)": (
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        "Dialoge aus Shakespeares Stücken, ca. 1,1 Mio. Zeichen. Der Klassiker – funktioniert sehr gut."),
    "Alice im Wunderland (Englisch)": (
        "https://www.gutenberg.org/cache/epub/11/pg11.txt",
        "Lewis Carroll, 1865, ca. 145.000 Zeichen. Eher kurz – achte auf Overfitting!"),
    "Sherlock Holmes (Englisch)": (
        "https://www.gutenberg.org/cache/epub/1661/pg1661.txt",
        "Arthur Conan Doyle, Detektivgeschichten, ca. 560.000 Zeichen."),
    "Grimms Märchen (Englisch)": (
        "https://www.gutenberg.org/cache/epub/2591/pg2591.txt",
        "Die Märchen der Brüder Grimm in englischer Übersetzung, ca. 520.000 Zeichen."),
    "Goethe – Faust I (Deutsch)": (
        "https://www.gutenberg.org/cache/epub/2229/pg2229.txt",
        "Theaterstück in Versen, ca. 190.000 Zeichen. Achte auf die Reime und Sprecher-Namen!"),
    "Kafka – Die Verwandlung (Deutsch)": (
        "https://www.gutenberg.org/cache/epub/22367/pg22367.txt",
        "Erzählung, ca. 120.000 Zeichen. Eher kurz – achte auf Overfitting!"),
}

CATEGORY_COLORS = {
    "Vokal": "#e15759", "Konsonant": "#4e79a7", "Großbuchstabe": "#f28e2b",
    "Ziffer": "#59a14f", "Leerzeichen / Umbruch": "#9c755f", "Satzzeichen / Sonstiges": "#b07aa1",
}


def _fmt(n):
    return f"{n:,}".replace(",", ".")


def _visible(ch):
    return {" ": "␣", "\n": "↵", "\t": "⇥"}.get(ch, ch)


def _decode_bytes(raw):
    for enc in ("utf-8-sig", "cp1252"):
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            pass
    return raw.decode("utf-8", errors="replace")


def _strip_gutenberg(text):
    start = re.search(r"\*\*\* ?START OF.*?\*\*\*", text)
    end = re.search(r"\*\*\* ?END OF", text)
    if start and end:
        text = text[start.end():end.start()]
    return text.strip()


def load_preset(name):
    url = PRESETS[name][0]
    cache = DATA_DIR / ("preset_" + url.rsplit("/", 1)[-1])
    if not cache.exists():
        print("Lade Datensatz herunter …")
        with urllib.request.urlopen(url, timeout=60) as response:
            raw = _decode_bytes(response.read())
        cache.write_text(_strip_gutenberg(raw), encoding="utf-8")
    return cache.read_text(encoding="utf-8")


class DatasetChooser:
    SOURCES = ["Beispiel-Datensatz", "Datei hochladen", "Datei aus data/-Ordner", "Text einfügen"]

    def __init__(self):
        self.source = widgets.ToggleButtons(options=self.SOURCES, description="Quelle:")
        self.preset = widgets.Dropdown(options=list(PRESETS), description="Datensatz:",
                                       layout=widgets.Layout(width="480px"))
        self.preset_info = widgets.HTML()
        self.upload = widgets.FileUpload(accept=".txt", multiple=False, description="Datei wählen")
        files = sorted(p.name for p in DATA_DIR.glob("*.txt") if not p.name.startswith("preset_"))
        self.local = widgets.Dropdown(options=files or ["(keine .txt-Dateien gefunden)"], description="Datei:",
                                      layout=widgets.Layout(width="480px"))
        self.paste = widgets.Textarea(
            placeholder="Füge hier deinen Text ein – je mehr, desto besser (mindestens ein paar Seiten) …",
            layout=widgets.Layout(width="100%", height="200px"))
        self.pages = {
            "Beispiel-Datensatz": widgets.VBox([self.preset, self.preset_info]),
            "Datei hochladen": widgets.VBox([
                widgets.HTML("Wähle eine <b>.txt</b>-Datei von deinem Computer aus:"), self.upload]),
            "Datei aus data/-Ordner": widgets.VBox([
                widgets.HTML(f"Lege eine <b>.txt</b>-Datei in den Ordner <code>{html.escape(str(DATA_DIR))}</code> "
                             "und führe diese Zelle danach erneut aus."), self.local]),
            "Text einfügen": self.paste,
        }
        self.box = widgets.VBox()
        self.source.observe(lambda _: self._refresh(), names="value")
        self.preset.observe(lambda _: self._refresh(), names="value")
        self._refresh()

    def _refresh(self):
        self.preset_info.value = f"<i>{PRESETS[self.preset.value][1]}</i>"
        self.box.children = [self.source, self.pages[self.source.value]]

    def show(self):
        display(self.box)

    def load(self):
        source = self.source.value
        if source == "Beispiel-Datensatz":
            name, text = self.preset.value, load_preset(self.preset.value)
        elif source == "Datei hochladen":
            if not self.upload.value:
                raise ValueError("Du hast noch keine Datei hochgeladen. Wähle oben eine Datei aus und führe diese Zelle erneut aus.")
            file = self.upload.value[0]
            name, text = file["name"], _decode_bytes(bytes(file["content"]))
        elif source == "Datei aus data/-Ordner":
            path = DATA_DIR / self.local.value
            if not path.is_file():
                raise ValueError(f"Keine Datei ausgewählt. Lege eine .txt-Datei in {DATA_DIR} und führe die Zelle oben erneut aus.")
            name, text = path.name, _decode_bytes(path.read_bytes())
        else:
            name, text = "Eigener Text", self.paste.value
        text = text.replace("\r\n", "\n").replace("\r", "\n")
        if len(text) < MIN_CHARS:
            raise ValueError(f"Der Text hat nur {_fmt(len(text))} Zeichen – das ist zu wenig zum Trainieren. "
                             f"Du brauchst mindestens {_fmt(MIN_CHARS)} Zeichen.")
        if len(text) > MAX_CHARS:
            print(f"Der Text ist sehr lang – es werden nur die ersten {_fmt(MAX_CHARS)} Zeichen verwendet.")
            text = text[:MAX_CHARS]
        return name, text


def show_text_stats(name, text):
    warning = ""
    if len(text) < 50_000:
        warning = ("<p style='color:#c0392b'>Der Text ist recht kurz. Das Modell wird ihn wahrscheinlich "
                   "<b>auswendig lernen</b> (Overfitting). Beobachte beim Training den Validierungs-Loss!</p>")
    display(HTML(
        f"<h4>{html.escape(name)}</h4><ul>"
        f"<li><b>{_fmt(len(text))}</b> Zeichen</li>"
        f"<li><b>{_fmt(len(text.split()))}</b> Wörter</li>"
        f"<li><b>{len(set(text))}</b> verschiedene Zeichen</li></ul>{warning}"
        f"<b>Vorschau:</b><pre style='background:#f6f8fa;color:#222;padding:8px;max-height:220px;overflow:auto;"
        f"white-space:pre-wrap'>{html.escape(text[:800])} …</pre>"))
    counts = Counter(text).most_common(30)
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.bar([_visible(c) for c, _ in counts], [n / len(text) * 100 for _, n in counts], color="#4e79a7")
    ax.set_title("Die 30 häufigsten Zeichen  (␣ = Leerzeichen, ↵ = Zeilenumbruch)")
    ax.set_ylabel("Anteil in %")
    plt.show()


def show_tokens(s, stoi):
    parts = []
    for ch in s:
        known = ch in stoi
        color = f"hsl({stoi[ch] * 47 % 360}, 70%, 85%)" if known else "#ff6b6b"
        token_id = stoi[ch] if known else "?"
        parts.append(
            f"<span style='display:inline-block;margin:2px;padding:2px 5px;border-radius:4px;background:{color};"
            f"color:#000;text-align:center;font-family:monospace'><b>{html.escape(_visible(ch))}</b>"
            f"<br><small>{token_id}</small></span>")
    display(HTML("".join(parts)))


def token_explorer(stoi, default):
    box = widgets.Text(value=default, description="Dein Text:", layout=widgets.Layout(width="600px"))
    out = widgets.Output()

    def update(_=None):
        with out:
            clear_output(wait=True)
            show_tokens(box.value, stoi)
    box.observe(update, names="value")
    update()
    display(box, out)


def show_training_example(x, y, decode):
    rows = "".join(
        f"<tr><td style='font-family:monospace;text-align:right'>{html.escape(decode(x[:t + 1].tolist()).replace(chr(10), '↵'))}</td>"
        f"<td>→</td><td style='font-family:monospace'><b>{html.escape(_visible(decode([y[t].item()])))}</b></td></tr>"
        for t in range(len(x)))
    display(HTML(f"<table><tr><th>Das Modell sieht (Kontext) …</th><th></th><th>… und soll vorhersagen (Ziel)</th></tr>{rows}</table>"))


CONFIG_PRESETS = {
    "Klein": dict(block_size=64, n_embd=64, n_head=4, n_layer=2, batch_size=32,
                    max_iters=2000, learning_rate=3e-3, dropout=0.0),
    "Mittel": dict(block_size=128, n_embd=128, n_head=4, n_layer=4, batch_size=32,
                     max_iters=3000, learning_rate=1e-3, dropout=0.1),
    "Groß (nur mit GPU)": dict(block_size=256, n_embd=384, n_head=6, n_layer=6, batch_size=64,
                                 max_iters=5000, learning_rate=3e-4, dropout=0.2),
}


class ConfigPanel:
    def __init__(self, device, vocab_size):
        self.vocab_size = vocab_size
        style = {"description_width": "160px"}
        layout = widgets.Layout(width="460px")
        self.preset = widgets.ToggleButtons(options=list(CONFIG_PRESETS), description="Voreinstellung:")
        self.w = {
            "block_size": widgets.Dropdown(options=[16, 32, 64, 128, 256], description="Kontextlänge"),
            "n_embd": widgets.Dropdown(options=[32, 48, 64, 96, 128, 192, 256, 384], description="Embedding-Größe"),
            "n_head": widgets.Dropdown(options=[1, 2, 3, 4, 6, 8], description="Attention-Köpfe"),
            "n_layer": widgets.IntSlider(min=1, max=8, description="Transformer-Blöcke"),
            "batch_size": widgets.Dropdown(options=[8, 16, 32, 64, 128], description="Batch-Größe"),
            "max_iters": widgets.IntSlider(min=100, max=10_000, step=100, description="Trainingsschritte"),
            "learning_rate": widgets.Dropdown(options=[("0.0001", 1e-4), ("0.0003", 3e-4), ("0.001", 1e-3),
                                                       ("0.003", 3e-3), ("0.01", 1e-2)], description="Lernrate"),
            "dropout": widgets.FloatSlider(min=0.0, max=0.5, step=0.05, description="Dropout"),
        }
        for w in self.w.values():
            w.style, w.layout = style, layout
            w.observe(self._update_info, names="value")
        self.info = widgets.HTML()
        self.preset.observe(self._apply_preset, names="value")
        self.preset.value = "Klein" if device == "cpu" else "Groß (nur mit GPU)"
        self._apply_preset()

    def _apply_preset(self, _=None):
        for key, value in CONFIG_PRESETS[self.preset.value].items():
            self.w[key].value = value

    def _update_info(self, _=None):
        c = {k: w.value for k, w in self.w.items()}
        if c["n_embd"] % c["n_head"]:
            self.info.value = (f"<b style='color:#c0392b'>Die Embedding-Größe ({c['n_embd']}) muss durch die Anzahl "
                               f"der Attention-Köpfe ({c['n_head']}) teilbar sein.</b>")
            return
        n, v = c["n_embd"], self.vocab_size
        params = 2 * v * n + c["block_size"] * n + c["n_layer"] * 12 * n * n
        self.info.value = f"Geschätzte Modellgröße: <b>≈ {_fmt(round(params, -3))} Parameter</b>"

    def show(self):
        display(widgets.VBox([self.preset, *self.w.values(), self.info]))

    def get(self):
        config = {k: w.value for k, w in self.w.items()}
        if config["n_embd"] % config["n_head"]:
            raise ValueError("Die Embedding-Größe muss durch die Anzahl der Attention-Köpfe teilbar sein. Ändere die Einstellungen oben.")
        return config


class TrainingMonitor:
    def __init__(self, baseline, total_steps):
        self.baseline, self.total_steps = baseline, total_steps
        self.plot = display(HTML("<i>Training startet …</i>"), display_id=True)
        self.status = display(HTML(""), display_id=True)

    def update(self, history, it, elapsed, sample):
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(history["step"], history["train"], "o-", label="Trainings-Loss")
        ax.plot(history["step"], history["val"], "o-", label="Validierungs-Loss")
        ax.axhline(self.baseline, ls="--", color="gray", label=f"Zufälliges Raten ({self.baseline:.2f})")
        ax.set_xlabel("Trainingsschritt")
        ax.set_ylabel("Loss  (kleiner = besser)")
        ax.set_ylim(0, max(self.baseline, max(history["train"] + history["val"])) * 1.1)
        ax.grid(alpha=0.3)
        ax.legend(loc="upper right")
        self.plot.update(fig)
        plt.close(fig)
        eta = elapsed / it * (self.total_steps - it) if it else 0
        self.status.update(HTML(
            f"<progress value='{it}' max='{self.total_steps}' style='width:400px'></progress> "
            f"Schritt <b>{it}/{self.total_steps}</b> · {elapsed:.0f} s vergangen · noch ca. {eta:.0f} s"
            f"<br><b>Aktueller Loss:</b> Training {history['train'][-1]:.3f} · Validierung {history['val'][-1]:.3f}"
            f"<br><b>So schreibt das Modell gerade:</b>"
            f"<pre style='background:#f6f8fa;color:#222;padding:8px;white-space:pre-wrap'>{html.escape(sample)}</pre>"))


def _highlight(prompt, result):
    return (f"<pre style='background:#f6f8fa;color:#222;padding:10px;white-space:pre-wrap;font-size:13px'>"
            f"<b style='background:#ffe08a'>{html.escape(prompt)}</b>{html.escape(result[len(prompt):])}</pre>")


def generation_playground(generate_fn, encode, decode, default_prompt):
    prompt = widgets.Textarea(value=default_prompt, description="Anfang:", layout=widgets.Layout(width="600px", height="60px"))
    length = widgets.IntSlider(value=300, min=50, max=1500, step=50, description="Länge:")
    temperature = widgets.FloatSlider(value=0.8, min=0.1, max=2.0, step=0.1, description="Temperatur:")
    button = widgets.Button(description="Text generieren", button_style="success")
    out = widgets.Output()

    def on_click(_):
        with out:
            clear_output()
            print("Das Modell schreibt …")
            result = generate_fn(prompt.value, length.value, temperature.value)
            clear_output()
            display(HTML(_highlight(decode(encode(prompt.value)), result)))
    button.on_click(on_click)
    display(widgets.VBox([prompt, length, temperature, button, out]))
    return button


def plot_next_char(model, encode, itos, prompt, temperature, top=15):
    ids = encode(prompt)[-model.block_size:]
    if not ids:
        print("Bitte gib einen Text ein (mit Zeichen, die im Vokabular vorkommen).")
        return
    dev = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        logits, _ = model(torch.tensor([ids], device=dev))
    model.train()
    probs = F.softmax(logits[0, -1] / temperature, dim=-1).cpu()
    p, idx = probs.topk(min(top, len(probs)))
    labels = [f"„{_visible(itos[i])}“" for i in idx.tolist()]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    bars = ax.barh(labels[::-1], (p * 100).tolist()[::-1], color="#4e79a7")
    ax.bar_label(bars, fmt="%.1f %%", padding=3)
    ax.set_xlim(0, 105)
    ax.set_xlabel("Wahrscheinlichkeit in %")
    shown = prompt[-25:].replace("\n", "↵")
    ax.set_title(f"Welches Zeichen kommt nach „…{shown}“?   (Temperatur {temperature})")
    plt.show()


def next_char_explorer(model, encode, itos, default_prompt):
    prompt = widgets.Text(value=default_prompt, description="Text:", continuous_update=False,
                          layout=widgets.Layout(width="600px"))
    temperature = widgets.FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1, description="Temperatur:",
                                      continuous_update=False)
    out = widgets.interactive_output(
        lambda p, t: plot_next_char(model, encode, itos, p, t), {"p": prompt, "t": temperature})
    display(widgets.VBox([prompt, temperature]), out)


def plot_attention(model, encode, itos, text, layer, head, max_len=40):
    ids = encode(text)[-min(max_len, model.block_size):]
    if len(ids) < 2:
        print("Bitte gib mindestens 2 Zeichen ein.")
        return
    dev = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        model(torch.tensor([ids], device=dev))
    model.train()
    heads = model.blocks[layer].sa.heads
    labels = [_visible(itos[i]) for i in ids]
    selected = range(len(heads)) if head == -1 else [head]
    cols = min(len(selected), 3)
    rows = math.ceil(len(selected) / cols)
    size = 7 if head != -1 else 4.5
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, h in zip(axes.flat, selected):
        ax.axis("on")
        ax.imshow(heads[h].last_attention[0].cpu(), cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(labels)), labels, fontsize=8)
        ax.set_yticks(range(len(labels)), labels, fontsize=8)
        ax.set_title(f"Block {layer + 1}, Kopf {h + 1}")
        if head != -1:
            ax.set_xlabel("… schaut auf dieses Zeichen")
            ax.set_ylabel("Dieses Zeichen …")
    fig.tight_layout()
    plt.show()


def attention_explorer(model, encode, itos, default_text):
    n_layer, n_head = len(model.blocks), len(model.blocks[0].sa.heads)
    text = widgets.Text(value=default_text, description="Text:", continuous_update=False,
                        layout=widgets.Layout(width="600px"))
    layer = widgets.IntSlider(value=1, min=1, max=n_layer, description="Block:", continuous_update=False)
    head = widgets.Dropdown(options=[("Alle Köpfe", -1)] + [(f"Kopf {i + 1}", i) for i in range(n_head)],
                            value=0, description="Kopf:")
    out = widgets.interactive_output(
        lambda t, l, h: plot_attention(model, encode, itos, t, l - 1, h), {"t": text, "l": layer, "h": head})
    display(widgets.VBox([text, widgets.HBox([layer, head])]), out)


def _category(ch):
    if ch.lower() in "aeiouäöüy" and ch.islower():
        return "Vokal"
    if ch.isalpha():
        return "Großbuchstabe" if ch.isupper() else "Konsonant"
    if ch.isdigit():
        return "Ziffer"
    if ch.isspace():
        return "Leerzeichen / Umbruch"
    return "Satzzeichen / Sonstiges"


def plot_embeddings(model, itos):
    weights = model.token_embedding.weight.detach().cpu().float().numpy()
    weights = weights - weights.mean(axis=0)
    _, _, vt = np.linalg.svd(weights, full_matrices=False)
    points = weights @ vt[:2].T
    fig, ax = plt.subplots(figsize=(10, 8))
    for i, (x, y) in enumerate(points):
        color = CATEGORY_COLORS[_category(itos[i])]
        ax.scatter(x, y, color=color, s=10)
        ax.annotate(_visible(itos[i]), (x, y), fontsize=12, color=color, ha="center", va="bottom")
    for name, color in CATEGORY_COLORS.items():
        ax.scatter([], [], color=color, label=name)
    ax.legend(loc="best")
    ax.set_title("Wo liegen die Zeichen im Embedding-Raum? (auf 2 Dimensionen reduziert)")
    ax.grid(alpha=0.3)
    plt.show()


print("Hilfsfunktionen geladen. Daten werden in", DATA_DIR, "gespeichert.")

## 1. Datensatz auswählen

Ein Sprachmodell kann nur so gut sein wie seine Trainingsdaten. **Du entscheidest, wovon dein Modell lernt:**

- **Beispiel-Datensatz** – klassische Bücher und Theaterstücke (werden automatisch heruntergeladen)
- **Datei hochladen** – eine eigene `.txt`-Datei, z. B. Songtexte, Gedichte, Rezepte, Code, Pokémon-Namen, Filmzitate …
- **Datei aus `data/`-Ordner** – eine `.txt`-Datei, die du in den Ordner `data/` dieses Projekts gelegt hast
- **Text einfügen** – Text direkt hineinkopieren

> **Tipps für einen guten Datensatz**
> - **Je mehr, desto besser.** Mindestens 10.000 Zeichen, ideal sind 200.000+ (ungefähr ein Roman).
> - **Einheitlicher Stil.** Z. B. nur Songtexte eines Genres oder nur Gedichte – nicht wild gemischt.
> - Das Modell lernt **Zeichen**, nicht Bedeutung. Texte mit klarer Struktur (Reime, Dialoge mit Namen, Code) liefern lustige Ergebnisse!

In [ ]:
auswahl = DatasetChooser()
auswahl.show()

Wenn du deine Wahl getroffen hast, führe die nächste Zelle aus. Sie lädt den Text und zeigt dir eine Übersicht:

In [ ]:
dataset_name, text = auswahl.load()
show_text_stats(dataset_name, text)

> **Aufgabe 1:** Schau dir die Statistik an. Wie viele *verschiedene* Zeichen gibt es? Welche Zeichen sind am häufigsten? Überrascht dich etwas?

**Deine Antwort:**

_…hier schreiben (Doppelklick auf diese Zelle)…_

## 2. Tokenisierung: Text → Zahlen

Neuronale Netze können nur mit Zahlen rechnen. Darum bekommt jedes Zeichen eine Nummer (**Token-ID**). Die Liste aller Zeichen, die das Modell kennt, heißt **Vokabular**.

> Große Modelle wie GPT-4 zerlegen Text nicht in einzelne Zeichen, sondern in **Wortteile** (z. B. `Trans` + `former`) und haben ein Vokabular von etwa 100.000 Tokens. Wir verwenden einzelne Zeichen – das ist einfacher und reicht für ein kleines Modell.

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}  # string → integer
itos = {i: ch for i, ch in enumerate(chars)}  # integer → string


def encode(s):
    """Text → Liste von Token-IDs (unbekannte Zeichen werden übersprungen)."""
    return [stoi[c] for c in s if c in stoi]


def decode(ids):
    """Liste von Token-IDs → Text."""
    return "".join(itos[i] for i in ids)


print(f"Vokabular: {vocab_size} verschiedene Zeichen")
print(repr("".join(chars)))
print()
beispiel = text[:20]
print(f"{beispiel!r}  →  {encode(beispiel)}")

Probier es aus! Tippe einen Text ein und sieh dir an, welche ID jedes Zeichen bekommt:

In [ ]:
token_explorer(stoi, default=text[:40].replace("\n", " "))

> **Aufgabe 2:** Tippe Wörter mit Umlauten, Emojis oder Sonderzeichen ein. Was bedeuten die **rot** markierten Zeichen? Was würde passieren, wenn du dem Modell so einen Text gibst?

**Deine Antwort:**

_…hier schreiben (Doppelklick auf diese Zelle)…_

## 3. Trainingsbeispiele bauen

Jetzt wandeln wir den ganzen Text in Zahlen um und teilen ihn auf:

- **90 % Trainingsdaten** – davon lernt das Modell.
- **10 % Validierungsdaten** – die sieht das Modell beim Lernen *nie*. Damit prüfen wir, ob es wirklich etwas *verstanden* oder nur *auswendig gelernt* hat. (Wie bei einer Schularbeit mit neuen Aufgaben statt den Übungsbeispielen.)

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Trainingsdaten:    {_fmt(len(train_data)):>9} Zeichen")
print(f"Validierungsdaten: {_fmt(len(val_data)):>9} Zeichen")

Das Modell bekommt immer einen **Ausschnitt** aus dem Text und soll für **jede Position** das nächste Zeichen vorhersagen.
So stecken in einem einzigen Ausschnitt gleich mehrere Trainingsbeispiele – mit immer mehr Kontext:

In [ ]:
ausschnitt = train_data[:9]
x, y = ausschnitt[:-1], ausschnitt[1:]  # Ziel = Text um ein Zeichen verschoben
show_training_example(x, y, decode)

Beim Training nehmen wir viele zufällige Ausschnitte gleichzeitig – einen **Batch**. Die GPU (oder CPU) kann sie alle parallel verarbeiten:

In [ ]:
def get_batch(split, block_size, batch_size):
    """Holt `batch_size` zufällige Ausschnitte der Länge `block_size`."""
    source = train_data if split == "train" else val_data
    starts = torch.randint(len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[s : s + block_size] for s in starts])
    y = torch.stack([source[s + 1 : s + block_size + 1] for s in starts])
    return x.to(device), y.to(device)


xb, yb = get_batch("train", block_size=8, batch_size=4)
print("Eingabe (x):\n", xb)
print("Ziel (y):\n", yb)

> **Aufgabe 3:** Ein Ausschnitt der Länge 8 enthält 8 Trainingsbeispiele. Was „weiß“ das Modell beim ersten Beispiel, was beim letzten? Bei welchem ist die Vorhersage wohl am schwierigsten?

**Deine Antwort:**

_…hier schreiben (Doppelklick auf diese Zelle)…_

## 4. Das Modell bauen

So ist unser Mini-GPT aufgebaut:

```
   "Hallo Wel"  →  Token-IDs  →  Embedding  +  Positions-Embedding
                                        │
                           ┌────────────▼────────────┐
                           │    Transformer-Block    │
                           │  ① Self-Attention       │  ← Zeichen „schauen“ auf frühere Zeichen
                           │  ② Feed-Forward         │  ← jedes Zeichen „denkt nach“
                           └────────────┬────────────┘
                                        │   (mehrmals hintereinander)
                                        ▼
                 Wahrscheinlichkeit für jedes mögliche nächste Zeichen:
                         „t“: 71 %   „l“: 12 %   „s“: 3 %   …
```

| Baustein | Was er macht |
|---|---|
| **Embedding** | Jede Token-ID wird zu einem Vektor (einer Liste von Zahlen). Das Modell lernt selbst, welche Zahlen gut sind. |
| **Positions-Embedding** | Sagt dem Modell, *wo* im Text ein Zeichen steht – sonst wäre „Hund beißt Mann“ = „Mann beißt Hund“. |
| **Self-Attention** | Das Herzstück! Jedes Zeichen stellt eine Frage (**Query**), jedes frühere Zeichen hat ein Etikett (**Key**). Passen Frage und Etikett zusammen, wird dessen Information (**Value**) übernommen. |
| **Maske** | Ein Zeichen darf nur auf *frühere* Zeichen schauen, nicht in die Zukunft – sonst wäre Vorhersagen ja geschummelt. |
| **Multi-Head** | Mehrere Attention-**Köpfe** arbeiten parallel und können auf verschiedene Dinge achten. |
| **Feed-Forward** | Ein kleines neuronales Netz, das die gesammelten Informationen für jedes Zeichen weiterverarbeitet. |
| **Residual + LayerNorm** | Technische Tricks, damit auch tiefe Netze stabil lernen. |

Hier ist der komplette Code – ein echter Transformer in ca. 80 Zeilen:

In [ ]:
class Head(nn.Module):
    """Ein einzelner Self-Attention-Kopf."""

    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.query = nn.Linear(n_embd, head_size, bias=False)  # "Wonach suche ich?"
        self.key = nn.Linear(n_embd, head_size, bias=False)    # "Was habe ich zu bieten?"
        self.value = nn.Linear(n_embd, head_size, bias=False)  # "Was gebe ich weiter?"
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
        self.last_attention = None  # wird für die Visualisierung gespeichert

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.query(x), self.key(x), self.value(x)
        # Wie gut passt jede Frage (Query) zu jedem Etikett (Key)?
        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        # Maske: Kein Blick in die Zukunft!
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        # Softmax macht daraus Gewichte, die zusammen 1 (= 100 %) ergeben
        weights = F.softmax(scores, dim=-1)
        self.last_attention = weights.detach()
        return self.dropout(weights) @ v


class MultiHeadAttention(nn.Module):
    """Mehrere Attention-Köpfe parallel, Ergebnisse werden zusammengefügt."""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    """Ein kleines neuronales Netz, das auf jedes Zeichen einzeln angewendet wird."""

    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Ein Transformer-Block: erst kommunizieren (Attention), dann nachdenken (Feed-Forward)."""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.sa = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))    # "+ x" = Residual-Verbindung
        x = x + self.ffwd(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd, n_head, n_layer, dropout):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)  # Vektor → eine Punktzahl pro Zeichen im Vokabular

    def forward(self, idx, targets=None):
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])        # nur die letzten block_size Zeichen
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)  # zufällig ziehen – gemäß Wahrscheinlichkeit
            idx = torch.cat([idx, next_id], dim=1)
        return idx

### Modellgröße einstellen

Jetzt legst du fest, wie groß dein Modell wird und wie lange es trainiert. Fang mit der Voreinstellung an und experimentiere später (siehe Challenges)!

| Einstellung | Bedeutung |
|---|---|
| **Kontextlänge** | Wie viele Zeichen das Modell maximal „zurückschauen“ kann. |
| **Embedding-Größe** | Wie viele Zahlen jedes Zeichen beschreiben. Größer = mehr „Ausdruckskraft“. |
| **Attention-Köpfe** | Wie viele Köpfe parallel auf den Text schauen. |
| **Transformer-Blöcke** | Wie viele Blöcke hintereinander. Mehr = tieferes Netz. |
| **Batch-Größe** | Wie viele Ausschnitte pro Trainingsschritt verarbeitet werden. |
| **Trainingsschritte** | Wie oft das Modell seine Gewichte anpasst. |
| **Lernrate** | Wie groß jede Anpassung ist. Zu groß → chaotisch, zu klein → lernt kaum. |
| **Dropout** | Schaltet beim Training zufällig Teile ab. Hilft gegen Auswendiglernen. |

> Auf einer **CPU** dauert „Klein“ ein paar Minuten. „Groß“ ist nur mit einer GPU sinnvoll.

In [ ]:
einstellungen = ConfigPanel(device, vocab_size)
einstellungen.show()

Führe die nächste Zelle aus, um das Modell mit diesen Einstellungen zu erstellen. **Wenn du die Einstellungen änderst, musst du sie erneut ausführen.**

In [ ]:
config = einstellungen.get()
torch.manual_seed(42)

model = MiniGPT(
    vocab_size=vocab_size,
    block_size=config["block_size"],
    n_embd=config["n_embd"],
    n_head=config["n_head"],
    n_layer=config["n_layer"],
    dropout=config["dropout"],
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])
history = {"step": [], "train": [], "val": []}

n_params = sum(p.numel() for p in model.parameters())
print(f"Dein Modell hat {n_params:,} Parameter.".replace(",", "."))
print("   Zum Vergleich: GPT-2 (2019) hatte 124.000.000, GPT-3 (2020) 175.000.000.000.")

Was schreibt das Modell, **bevor** es trainiert wurde?

In [ ]:
def generate_text(prompt="", length=300, temperature=1.0):
    """Lässt das Modell nach `prompt` weitere `length` Zeichen schreiben."""
    ids = encode(prompt) or [stoi.get("\n", 0)]
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    model.eval()
    out = model.generate(idx, length, temperature)
    model.train()
    return decode(out[0].tolist())


print(generate_text(length=300))

Totaler Zeichensalat – logisch, alle Gewichte im Modell sind noch **zufällig**. Das ändern wir jetzt!

## 5. Training

In jedem Trainingsschritt passiert Folgendes:
1. Einen Batch zufälliger Textausschnitte holen.
2. Das Modell sagt für jede Position das nächste Zeichen vorher.
3. Der **Loss** misst, wie falsch die Vorhersagen sind – genauer: wie „überrascht“ das Modell vom richtigen Zeichen ist.
4. **Backpropagation** berechnet, in welche Richtung jedes der Gewichte verändert werden muss, damit der Loss kleiner wird.
5. Der **Optimizer** passt alle Gewichte ein kleines Stück an.

**So liest du das Diagramm:**
- Die graue Linie zeigt den Loss bei **zufälligem Raten**. Da startet das Modell.
- Sinken **beide** Kurven → das Modell lernt.
- Sinkt der Trainings-Loss, aber der Validierungs-Loss **steigt** → das Modell lernt auswendig (**Overfitting**).

> Du kannst das Training jederzeit mit dem **Stopp-Knopf** (Interrupt Kernel) abbrechen – das Modell behält, was es bis dahin gelernt hat.
> Führst du die Zelle **nochmal** aus, trainiert das Modell **weiter**. Für einen Neustart führe die Zelle „Modell erstellen“ erneut aus.

In [ ]:
max_iters = config["max_iters"]
eval_interval = max(max_iters // 20, 1)  # 20 Messpunkte im Diagramm
eval_iters = 20 if device == "cpu" else 50


@torch.no_grad()
def estimate_loss():
    """Misst den durchschnittlichen Loss auf Trainings- und Validierungsdaten."""
    model.eval()
    result = {}
    for split in ("train", "val"):
        losses = [model(*get_batch(split, config["block_size"], config["batch_size"]))[1].item()
                  for _ in range(eval_iters)]
        result[split] = sum(losses) / len(losses)
    model.train()
    return result


def record(step):
    losses = estimate_loss()
    history["step"].append(step)
    history["train"].append(losses["train"])
    history["val"].append(losses["val"])


monitor = TrainingMonitor(baseline=math.log(vocab_size), total_steps=max_iters)
start_step = history["step"][-1] if history["step"] else 0
if not history["step"]:
    record(0)
start_time = time.time()

try:
    for it in range(1, max_iters + 1):
        xb, yb = get_batch("train", config["block_size"], config["batch_size"])
        logits, loss = model(xb, yb)            # 1.+2. vorhersagen, 3. Loss berechnen
        optimizer.zero_grad(set_to_none=True)
        loss.backward()                         # 4. Backpropagation
        optimizer.step()                        # 5. Gewichte anpassen

        if it % eval_interval == 0 or it == max_iters:
            record(start_step + it)
            monitor.update(history, it, time.time() - start_time, generate_text(length=200))
    print(f"Training fertig nach {time.time() - start_time:.0f} Sekunden!")
except KeyboardInterrupt:
    print("Training abgebrochen – das Modell behält, was es bis jetzt gelernt hat.")

> **Aufgabe 4:** Beobachte die Beispieltexte während des Trainings.
> - In welcher **Reihenfolge** lernt das Modell? (Häufige Buchstaben? Leerzeichen? Wörter? Satzzeichen? Großschreibung? Zeilenumbrüche?)
> - Notiere den Loss am **Anfang** und am **Ende**. Wie weit ist er unter „Zufälliges Raten“ gefallen?
> - Gibt es Anzeichen für Overfitting?

**Deine Antwort:**

_…hier schreiben (Doppelklick auf diese Zelle)…_

## 6. Text generieren

Jetzt darf dein Modell schreiben! Gib einen Anfang vor (gelb markiert) und lass es weiterschreiben.

**Die Temperatur** bestimmt, wie „mutig“ das Modell ist:
- **niedrig (0.1 – 0.5)** → wählt fast immer das wahrscheinlichste Zeichen: sicher, aber oft langweilig und wiederholend
- **mittel (0.7 – 1.0)** → ausgewogen
- **hoch (1.5 – 2.0)** → wählt auch unwahrscheinliche Zeichen: kreativ, aber chaotisch

In [ ]:
generation_playground(generate_text, encode, decode, default_prompt=text[:30])

### Was denkt das Modell?

Das Modell „schreibt“ nicht wirklich – es berechnet für **jedes** Zeichen im Vokabular eine Wahrscheinlichkeit und **würfelt** dann. Hier siehst du diese Wahrscheinlichkeiten. Ändere den Text und die Temperatur (mit Enter bestätigen):

In [ ]:
next_char_explorer(model, encode, itos, default_prompt=text[:30])

> **Aufgabe 5:** Generiere Texte mit Temperatur **0.2**, **1.0** und **2.0**. Beschreibe die Unterschiede. Schau dir im Diagramm an, was die Temperatur mit den Wahrscheinlichkeiten macht.
>
> **Aufgabe 6:** Finde einen Textanfang, bei dem sich das Modell **sehr sicher** ist (über 90 %), und einen, bei dem es **unsicher** ist. Warum ist das so?

**Deine Antwort:**

_…hier schreiben (Doppelklick auf diese Zelle)…_

## 7. Ins Modell hineinschauen

### Attention: Worauf achtet das Modell?

Jede **Zeile** ist ein Zeichen. Die Farbe zeigt, wie stark dieses Zeichen auf die früheren Zeichen (Spalten) „schaut“. Dunkelblau = viel Aufmerksamkeit.
Das leere Dreieck rechts oben ist die **Maske**: kein Blick in die Zukunft.

In [ ]:
attention_explorer(model, encode, itos, default_text=text[:30])

> **Aufgabe 7:** Wähle verschiedene Blöcke und Köpfe aus (oder „Alle Köpfe“).
> - Findest du einen Kopf, der fast nur auf das **direkt vorherige** Zeichen schaut?
> - Findest du einen Kopf, der auf **Leerzeichen** oder **Wortanfänge** achtet?

**Deine Antwort:**

_…hier schreiben (Doppelklick auf diese Zelle)…_

### Embeddings: Was hat das Modell über Zeichen gelernt?

Jedes Zeichen wird im Modell durch einen Vektor mit vielen Zahlen dargestellt. Wir quetschen diese Vektoren auf 2 Dimensionen zusammen, damit wir sie zeichnen können. Zeichen, die das Modell als **ähnlich** ansieht, liegen nahe beieinander.

In [ ]:
plot_embeddings(model, itos)

> **Aufgabe 8:** Welche Zeichen liegen nahe beieinander? Bilden Vokale, Großbuchstaben, Ziffern oder Satzzeichen Gruppen?
> Bedenke: Niemand hat dem Modell gesagt, was ein Vokal ist. Woher „weiß“ es das?

**Deine Antwort:**

_…hier schreiben (Doppelklick auf diese Zelle)…_

## 8. Modell speichern

Speichere dein trainiertes Modell, damit du es später wieder verwenden kannst. Die Datei landet im Ordner `data/`.

In [ ]:
dateiname = re.sub(r"\W+", "_", dataset_name).strip("_").lower() or "modell"
pfad = DATA_DIR / f"mini_gpt_{dateiname}.pt"
torch.save({"model": model.state_dict(), "config": config, "chars": chars, "history": history}, pfad)
print(f"Gespeichert unter: {pfad}  ({pfad.stat().st_size / 1e6:.1f} MB)")

# So lädst du das Modell später wieder (nachdem Schritt 0, 2 und die Modell-Klassen ausgeführt wurden):
# checkpoint = torch.load(pfad)
# model = MiniGPT(len(checkpoint["chars"]), **{k: checkpoint["config"][k] for k in
#                 ("block_size", "n_embd", "n_head", "n_layer", "dropout")}).to(device)
# model.load_state_dict(checkpoint["model"])

## 9. Challenges

Jetzt bist du dran! Wähle mindestens **zwei** Challenges aus. Für jede Challenge: Einstellungen ändern → „Modell erstellen“ ausführen → trainieren → Ergebnis notieren.

**1. Datensatz-Duell**
Trainiere ein Modell auf einem ganz anderen Datensatz (eigene Datei!). Vergleiche die generierten Texte. Was hat das Modell jeweils über den Stil gelernt?

**2. Größer = besser?**
Trainiere mit verschiedenen Modellgrößen und fülle die Tabelle aus:

| Konfiguration | Parameter | Trainingszeit | Train-Loss | Val-Loss | Eindruck vom Text |
|---|---|---|---|---|---|
| Klein | | | | | |
| eigene Einstellung: … | | | | | |
| eigene Einstellung: … | | | | | |

**3. Wie wichtig ist der Kontext?**
Trainiere einmal mit Kontextlänge **16** und einmal mit **128** (sonst gleiche Einstellungen). Was ändert sich am generierten Text?

**4. Lernrate ausreizen**
Probiere die Lernrate **0.0001** und **0.01**. Wie sieht die Loss-Kurve jeweils aus?

**5. Overfitting provozieren**
Nimm einen **kurzen** Text (z. B. ein paar Songtexte, ~15.000 Zeichen), ein großes Modell und viele Trainingsschritte, Dropout 0. Was passiert mit dem Validierungs-Loss? Was generiert das Modell? Und was hilft dagegen?

**6. Bonus für Profis**
Ändere die Funktion `generate` im Modell so, dass nur aus den **k wahrscheinlichsten** Zeichen gezogen wird (*Top-k-Sampling*). Tipp: `torch.topk`.

**Deine Antwort:**

_…hier schreiben (Doppelklick auf diese Zelle)…_

### Reflexion

1. **Versteht** dein Modell, was es schreibt? Begründe deine Antwort.
2. Was unterscheidet dein Mini-GPT von ChatGPT? (Denk an Größe, Datenmenge, Tokens – und daran, dass ChatGPT auf Fragen *antwortet*, statt nur Text fortzusetzen.)
3. Dein Modell gibt wieder, was in den Trainingsdaten steht. Welche Probleme kann das bei großen Sprachmodellen verursachen, die mit Texten aus dem Internet trainiert werden? (Stichwörter: Vorurteile, Falschinformationen, Urheberrecht)

**Deine Antwort:**

_…hier schreiben (Doppelklick auf diese Zelle)…_